# 03 - DRC Netting

Synthetic engineering and validation evidence, not final regulatory capital.

Verify same-obligor netting and seniority constraints after showing the public capital entrypoint.

Use this notebook as a companion to [`PACKAGE_JOURNEY.md`](../docs/PACKAGE_JOURNEY.md) and [`examples/run_demo.py`](../examples/run_demo.py). The happy path is the public package API: construct DRC positions and a calculation context, then call `frtb_drc.calculate_drc_capital`.

## Raw inputs your upstream risk engine must emit

A DRC upstream engine should emit one row per default-risk position with stable position and source-row identifiers, desk/legal-entity scope, risk class, instrument type, issuer or tranche/index identifiers where applicable, bucket key, seniority or credit quality, notional or market value, maturity, currency, lineage fields, and citation identifiers. The non-securitisation table contract is documented in [`docs/schemas/input_table/frtb_drc.nonsec.schema.json`](../../../docs/schemas/input_table/frtb_drc.nonsec.schema.json); package-specific context and fixture notes live in [`DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md).


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
for path in (ROOT, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from examples.drc_nonsec_fixture import load_drc_nonsec_v2_fixture, run_fixture_workflow
from frtb_drc.demo_data import ALL_POSITIONS, DEMO_CONTEXT

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def markdown_table(headers: list[str], rows: list[list[str]]) -> Markdown:
    header = "| " + " | ".join(headers) + " |"
    separator = "| " + " | ".join(["---"] * len(headers)) + " |"
    body = ["| " + " | ".join(row) + " |" for row in rows]
    return Markdown("\n".join([header, separator, *body]))


fixture = load_drc_nonsec_v2_fixture()
positions = fixture.positions
context = fixture.context
expected = fixture.expected_outputs
display(
    Markdown(
        f"Loaded fixture `{expected['fixture_id']}`: "
        f"`{len(positions)}` positions, "
        f"profile `{context.profile_id}`, "
        f"as-of `{context.calculation_date}`."
    )
)

## Public API happy path

The primary user contract is public API first: pass validated `DrcPosition` records and `DrcCalculationContext` to `calculate_drc_capital`. The implementation-detail sections below inspect intermediate mechanics for validation and auditability.


In [ ]:
from frtb_drc import calculate_drc_capital

public_result = calculate_drc_capital(positions, context=context)
print(f"Total DRC capital: {public_result.total_drc:,.2f}")
print(f"Input hash       : {public_result.input_hash}")
print(f"Profile hash     : {public_result.profile_hash}")


## Implementation details / math verification - Netting mechanics


In [ ]:
from frtb_drc.gross_jtd import calculate_gross_jtds
from frtb_drc.maturity import scale_gross_jtds
from frtb_drc.netting import NettingInput, calculate_net_jtds
from frtb_drc.data_models import DrcSeniority

gross_jtds = calculate_gross_jtds(positions)
scaled_jtds = scale_gross_jtds(
    ((g, p.maturity_years) for g, p in zip(gross_jtds, positions))
)
pos_by_id = {p.position_id: p for p in positions}
gross_by_id = {g.position_id: g for g in gross_jtds}
scaled_by_id = {s.position_id: s for s in scaled_jtds}

netting_inputs = tuple(
    NettingInput(
        gross_jtd=gross_by_id[p.position_id],
        scaled_jtd=scaled_by_id[p.position_id],
        seniority=DrcSeniority(p.seniority),
    )
    for p in positions
    if p.seniority is not None
)
net_jtds = calculate_net_jtds(netting_inputs)

rows = [
    [
        net.net_jtd_id,
        net.bucket_key,
        net.net_direction.value,
        f"{net.net_amount:>14,.2f}",
        str(len(net.rejected_offsets)),
    ]
    for net in net_jtds
]
display(markdown_table(["Net JTD ID", "Bucket", "Direction", "Net amount", "Rejected offsets"], rows))

## Accepted offsets — acme-corp

In [ ]:
# acme-corp: LONG SENIOR_DEBT (rank 1), SHORT NON_SENIOR_DEBT (rank 2)
# short_rank(2) >= long_rank(1) → ACCEPTED
acme_net = next(n for n in net_jtds if "acme" in n.net_jtd_id and n.net_direction.value == "LONG")
acme_long_p = pos_by_id["corp-acme-sr-l-001"]
acme_short_p = pos_by_id["corp-acme-nsr-s-001"]
print(f"LONG  seniority: {acme_long_p.seniority}  (rank 1)")
print(f"SHORT seniority: {acme_short_p.seniority}  (rank 2)")
print(f"short_rank(2) >= long_rank(1) → offset ACCEPTED")
print()
print(f"Scaled LONG  = {scaled_by_id['corp-acme-sr-l-001'].scaled_jtd:,.2f}")
print(f"Scaled SHORT = {scaled_by_id['corp-acme-nsr-s-001'].scaled_jtd:,.2f}")
print(f"Net amount   = {acme_net.net_amount:,.2f}  direction={acme_net.net_direction.value}")
print(f"Rejected offsets: {acme_net.rejected_offsets}")

## Rejected offsets — eta-finance

`eta-finance` has LONG NON\_SENIOR\_DEBT (rank 2) and SHORT SENIOR\_DEBT (rank 1).
`short_rank(1) < long_rank(2)` → the short is **too senior** to offset the long.
Both survive as separate net positions.

In [ ]:
eta_nets = [n for n in net_jtds if "eta-finance" in n.net_jtd_id]
for net in eta_nets:
    print(f"  {net.net_jtd_id}: direction={net.net_direction.value}  amount={net.net_amount:,.2f}")
    for ro in net.rejected_offsets:
        print(f"    rejected: {ro.reason_code}")
print()
print("Interpretation:")
print("  eta-finance LONG NSR (rank 2) cannot be hedged by SENIOR_DEBT short (rank 1)")
print("  → both positions carry through to bucket capital step")

## Rejected offsets — freddie-mac (PSE\_GSE bucket)

`freddie-mac` has LONG GSE\_ISSUED\_NOT\_GUARANTEED (rank 1) and SHORT GSE\_GUARANTEED (rank 0).
`short_rank(0) < long_rank(1)` → the short is **more senior** than the long → **rejected**.

In [ ]:
freddie_nets = [n for n in net_jtds if "freddie" in n.net_jtd_id]
for net in freddie_nets:
    print(f"  {net.net_jtd_id}: direction={net.net_direction.value}  amount={net.net_amount:,.2f}")
    for ro in net.rejected_offsets:
        print(f"    rejected: {ro.reason_code}")

## Netting: long vs short survivors by bucket

In [ ]:
import matplotlib.pyplot as plt

buckets = ["CORPORATE", "NON_US_SOVEREIGN", "PSE_GSE", "DEFAULTED"]
colours = ["#2563EB", "#059669", "#D97706", "#DC2626"]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, bucket, colour in zip(axes, buckets, colours):
    longs = [n.net_amount / 1e6 for n in net_jtds
             if n.bucket_key == bucket and n.net_direction.value == "LONG"]
    shorts = [n.net_amount / 1e6 for n in net_jtds
              if n.bucket_key == bucket and n.net_direction.value == "SHORT"]
    ax.bar(["LONG", "SHORT"], [sum(longs), sum(shorts)], color=colour, alpha=0.8)
    ax.set_title(bucket, fontsize=9)
    ax.set_ylabel("Net JTD (USD M)")
fig.suptitle("Net JTD totals by bucket and direction", fontsize=11)
fig.tight_layout()
plt.show()

## See also

- [`packages/frtb-drc/docs/PACKAGE_JOURNEY.md`](../docs/PACKAGE_JOURNEY.md) for the package workflow and maturity path.
- [`packages/frtb-drc/docs/DATASET_CONTRACT.md`](../docs/DATASET_CONTRACT.md) for fixture and input-shape notes.
- [`docs/CLIENT_INTEGRATION.md`](../../../docs/CLIENT_INTEGRATION.md) for suite-level client integration guidance.
- Sibling component journeys: [`frtb-sbm`](../../frtb-sbm/docs/PACKAGE_JOURNEY.md), [`frtb-rrao`](../../frtb-rrao/docs/PACKAGE_JOURNEY.md), [`frtb-cva`](../../frtb-cva/docs/PACKAGE_JOURNEY.md), [`frtb-ima`](../../frtb-ima/docs/PACKAGE_JOURNEY.md), and [`frtb-orchestration`](../../frtb-orchestration/docs/PACKAGE_JOURNEY.md).
